# Position Offset Evaluation

f5c eventalign reports 0-based positions of the **first base** of the k-mer.
Known modification sites are annotated as **1-based biological positions**.

When matching ground truth to pipeline output, we need:
```
pipeline_pos = biological_pos - offset
```

This notebook systematically evaluates offsets 0–5 to determine which
gives the best match between known modification sites and baleen's
modification scores.

### Why offset matters

For a 5-mer reported at position `p` (0-based, first base):
- Offset 0: match at first base of k-mer
- Offset 1: 1-based → 0-based conversion only
- Offset 2: center-ish of 5-mer (base index 2)
- Offset 3: current default in benchmarking notebooks
- Offset 4–5: for wider k-mers or different conventions

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.metrics import roc_curve, auc, average_precision_score

from baleen.eventalign import load_results
from baleen.eventalign._hierarchical import compute_sequential_modification_probabilities

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('Imports OK')

## 2. Configuration

In [ ]:
BALEEN_PKL = Path('../output/pipeline_results.pkl')

# Offsets to evaluate
OFFSETS_TO_TEST = [0, 1, 2, 3, 4, 5]

# Output
OUT_DIR = Path('figures_offset_eval')
OUT_DIR.mkdir(exist_ok=True)

print('Configuration set.')

## 3. Known Modifications (1-based biological positions)

These are the literature-confirmed E. coli rRNA modification sites, in **1-based** coordinates.

In [ ]:
KNOWN_MODIFICATIONS = {
    # Pseudouridine
    ('ecoli16S', 516):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 746):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 955):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 1911): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 1917): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2457): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2504): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2580): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2604): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2605): ('\u03a8',     'pseudouridine'),
    # N2-methylguanosine
    ('ecoli16S', 966):  ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1207): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1516): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 1835): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 2445): ('m2G',   'N2-methylguanosine'),
    # 5-methylcytidine
    ('ecoli16S', 967):  ('m5C',   '5-methylcytidine'),
    ('ecoli16S', 1407): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 1962): ('m5C',   '5-methylcytidine'),
    # 5-methyluridine
    ('ecoli23S', 747):  ('m5U',   '5-methyluridine'),
    ('ecoli23S', 1939): ('m5U',   '5-methyluridine'),
    # N6,N6-dimethyladenosine
    ('ecoli16S', 1518): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli16S', 1519): ('m6,6A', 'N6,N6-dimethyladenosine'),
    # N6-methyladenosine
    ('ecoli23S', 1618): ('m6A',   'N6-methyladenosine'),
    ('ecoli23S', 2030): ('m6A',   'N6-methyladenosine'),
    # N7-methylguanosine
    ('ecoli16S', 527):  ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2069): ('m7G',   'N7-methylguanosine'),
    # Other
    ('ecoli23S', 2498): ('Cm',    \"2'-O-methylcytosine\"),
    ('ecoli23S', 2449): ('D',     'dihydrouridine'),
    ('ecoli23S', 2251): ('Gm',    \"2'-O-methylguanosine\"),
    ('ecoli23S', 2552): ('Um',    \"2'-O-methyluridine\"),
    ('ecoli23S', 2501): ('ho5C',  '5-hydroxycytidine'),
    ('ecoli23S', 745):  ('m1G',   '1-methylguanosine'),
    ('ecoli23S', 2503): ('m2A',   '2-methyladenosine'),
    ('ecoli23S', 1915): ('m3\u03a8',   '3-methylpseudouridine'),
    ('ecoli16S', 1498): ('m3U',   '3-methyluridine'),
    ('ecoli16S', 1402): ('m4Cm',  \"N4,2'-O-dimethylcytidine\"),
}

print(f'Total known modification sites: {len(KNOWN_MODIFICATIONS)}')

## 4. Load Pipeline Results and Run Hierarchical Stages

In [ ]:
contig_results = None
for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        print(f'Loading: {candidate}')
        contig_results, metadata = load_results(candidate)
        break

if contig_results is None:
    raise FileNotFoundError('pipeline_results.pkl not found.')

print(f'Loaded {len(contig_results)} contigs')
for contig, cr in contig_results.items():
    positions = sorted(cr.positions.keys())
    print(f'  {contig}: {len(positions):,} positions, range [{min(positions)}..{max(positions)}]')

print('\nRunning hierarchical pipeline (V1 \u2192 V3)...')
hier_results = {}
for contig, cr in contig_results.items():
    hier_results[contig] = compute_sequential_modification_probabilities(cr)
    print(f'  {contig}: done')
print('Done.')

## 5. Inspect k-mer Length

Check what k-mer size f5c eventalign used, since this determines which offsets are meaningful.

In [ ]:
kmer_lengths = set()
for contig, cr in contig_results.items():
    for pos, pr in list(cr.positions.items())[:50]:
        if pr.reference_kmer:
            kmer_lengths.add(len(pr.reference_kmer))

print(f'K-mer lengths found: {kmer_lengths}')
KMER_LEN = max(kmer_lengths) if kmer_lengths else 5
print(f'Using k-mer length: {KMER_LEN}')
print(f'  Offset 0 = first base of k-mer (eventalign convention)')
print(f'  Offset 1 = 1-based to 0-based only')
print(f'  Offset {KMER_LEN // 2} = center of k-mer')
print(f'  Offset {KMER_LEN - 1} = last base of k-mer')

## 6. Evaluate Each Offset

For each candidate offset, convert known modification positions to pipeline coordinates, compute labels, and measure ROC-AUC / PR-AUC across all baleen scoring methods.

In [ ]:
SCORE_METHODS = [
    ('v1_z',          'V1 z-score'),
    ('v2_raw',        'V2 raw'),
    ('v2_knn',        'V2 kNN'),
    ('v3_hmm',        'V3 HMM'),
    ('stoichiometry', 'Stoichiometry'),
]

def build_site_df(contig_results, hier_results, known_mods, offset):
    """Build site-level DataFrame with y_true labels for a given offset."""
    # Convert biological 1-based to pipeline 0-based with offset
    known_set = {
        (contig, bio_pos - offset)
        for (contig, bio_pos) in known_mods.keys()
    }
    known_map = {
        (contig, bio_pos - offset): (mod_short, mod_full)
        for (contig, bio_pos), (mod_short, mod_full) in known_mods.items()
    }
    
    records = []
    for contig, cr in contig_results.items():
        cmr = hier_results.get(contig)
        for pos, pr in cr.positions.items():
            ps = cmr.position_stats.get(pos) if cmr else None
            is_mod = (contig, pos) in known_set
            
            rec = {
                'contig': contig,
                'position': pos,
                'kmer': pr.reference_kmer,
                'y_true': int(is_mod),
            }
            
            if ps is not None:
                ns = slice(0, ps.n_native)
                rec['v1_z']   = float(np.mean(ps.z_scores[ns]))
                rec['v2_raw'] = float(np.nanmean(ps.p_mod_raw[ns]))
                rec['v2_knn'] = float(np.nanmean(ps.p_mod_knn[ns]))
                rec['v3_hmm'] = float(np.nanmean(ps.p_mod_hmm[ns]))
                hmm_nat = ps.p_mod_hmm[ns]
                hmm_valid = hmm_nat[~np.isnan(hmm_nat)]
                rec['stoichiometry'] = float(np.mean(hmm_valid > 0.5)) if len(hmm_valid) > 0 else np.nan
            else:
                for col in ['v1_z','v2_raw','v2_knn','v3_hmm','stoichiometry']:
                    rec[col] = np.nan
            
            records.append(rec)
    
    return pd.DataFrame(records)


# Evaluate all offsets
offset_results = []
offset_dfs = {}

for offset in OFFSETS_TO_TEST:
    df = build_site_df(contig_results, hier_results, KNOWN_MODIFICATIONS, offset)
    offset_dfs[offset] = df
    y = df['y_true'].values
    n_pos = y.sum()
    
    # How many known sites actually landed on a pipeline position?
    n_known_total = len(KNOWN_MODIFICATIONS)
    
    for col, label in SCORE_METHODS:
        scores = df[col].values
        valid = ~np.isnan(scores)
        if len(np.unique(y[valid])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y[valid], scores[valid])
        roc_auc = auc(fpr, tpr)
        pr_auc = average_precision_score(y[valid], scores[valid])
        offset_results.append({
            'offset': offset,
            'method': label,
            'roc_auc': roc_auc,
            'pr_auc': pr_auc,
            'n_matched': n_pos,
            'n_known_total': n_known_total,
            'match_rate': n_pos / n_known_total,
        })

results_df = pd.DataFrame(offset_results)
print(results_df.to_string(index=False))

## 7. Match Rate: How Many Known Sites Hit a Pipeline Position?

If the offset is wrong, known modification sites won't overlap with any pipeline position, resulting in zero matches.

In [ ]:
match_summary = results_df.groupby('offset')[['n_matched', 'n_known_total', 'match_rate']].first()
print('Known site match rate by offset:')
print(match_summary.to_string())
print()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(match_summary.index, match_summary['n_matched'],
       color='#42A5F5', edgecolor='white')
ax.axhline(match_summary['n_known_total'].iloc[0], color='red',
           ls='--', lw=1.5, label=f'Total known sites ({match_summary["n_known_total"].iloc[0]})')
ax.set_xlabel('Position Offset')
ax.set_ylabel('# Known Sites Matched')
ax.set_title('Known Modification Sites Matched at Each Offset')
ax.set_xticks(OFFSETS_TO_TEST)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'match_rate_by_offset.pdf', bbox_inches='tight')
plt.show()

## 8. ROC-AUC and PR-AUC by Offset

Compare classification performance across offsets for each scoring method.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

method_colors = {
    'V1 z-score':    '#CE93D8',
    'V2 raw':        '#80CBC4',
    'V2 kNN':        '#29B6F6',
    'V3 HMM':        '#EF5350',
    'Stoichiometry': '#FF7043',
}

for ax, metric, title in [(axes[0], 'roc_auc', 'ROC-AUC'),
                           (axes[1], 'pr_auc', 'PR-AUC')]:
    for method in results_df['method'].unique():
        sub = results_df[results_df['method'] == method]
        ax.plot(sub['offset'], sub[metric], 'o-', lw=2, ms=7,
                color=method_colors.get(method, 'gray'),
                label=method)
    
    # Mark the best offset for V3 HMM
    hmm_sub = results_df[results_df['method'] == 'V3 HMM']
    if len(hmm_sub) > 0:
        best_idx = hmm_sub[metric].idxmax()
        best_off = hmm_sub.loc[best_idx, 'offset']
        best_val = hmm_sub.loc[best_idx, metric]
        ax.annotate(f'best={best_off}',
                    xy=(best_off, best_val),
                    xytext=(best_off + 0.3, best_val - 0.05),
                    arrowprops=dict(arrowstyle='->', color='red'),
                    fontsize=10, color='red', fontweight='bold')
    
    ax.set_xlabel('Position Offset')
    ax.set_ylabel(title)
    ax.set_title(f'{title} vs Position Offset')
    ax.set_xticks(OFFSETS_TO_TEST)
    ax.legend(fontsize=9)
    ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(OUT_DIR / 'auc_vs_offset.pdf', bbox_inches='tight')
plt.show()

## 9. Heatmap: Method x Offset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, title in [(axes[0], 'roc_auc', 'ROC-AUC'),
                           (axes[1], 'pr_auc', 'PR-AUC')]:
    pivot = results_df.pivot(index='method', columns='offset', values=metric)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.3, vmax=1.0, linewidths=0.5, ax=ax,
                cbar_kws={'label': title})
    ax.set_title(f'{title} by Method and Offset')
    ax.set_xlabel('Position Offset')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig(OUT_DIR / 'heatmap_method_offset.pdf', bbox_inches='tight')
plt.show()

## 10. Per-Site Diagnostic: Which Known Sites Are Found at Each Offset?

Show which specific known modification sites are matched (i.e., overlap with a pipeline position) at each offset.

In [ ]:
all_pipeline_positions = set()
for contig, cr in contig_results.items():
    for pos in cr.positions:
        all_pipeline_positions.add((contig, pos))

print(f'Total pipeline positions: {len(all_pipeline_positions):,}\n')

site_match_records = []
for (contig, bio_pos), (mod_short, mod_full) in sorted(KNOWN_MODIFICATIONS.items()):
    rec = {'contig': contig, 'bio_pos': bio_pos, 'mod': mod_short}
    for offset in OFFSETS_TO_TEST:
        pipeline_pos = bio_pos - offset
        rec[f'off{offset}'] = '\u2713' if (contig, pipeline_pos) in all_pipeline_positions else '\u2717'
    site_match_records.append(rec)

match_df = pd.DataFrame(site_match_records)
print(match_df.to_string(index=False))

# Count hits per offset
print('\n--- Hits per offset ---')
for offset in OFFSETS_TO_TEST:
    col = f'off{offset}'
    n_hit = (match_df[col] == '\u2713').sum()
    print(f'  offset={offset}: {n_hit}/{len(match_df)} sites matched')

## 11. Best Offset Summary

In [ ]:
print('=' * 70)
print('POSITION OFFSET EVALUATION SUMMARY')
print('=' * 70)

# Best offset per method
for metric in ['roc_auc', 'pr_auc']:
    print(f'\nBest offset by {metric.upper()}:')
    for method in results_df['method'].unique():
        sub = results_df[results_df['method'] == method]
        best = sub.loc[sub[metric].idxmax()]
        print(f'  {method:18s}  offset={int(best["offset"])}  {metric}={best[metric]:.4f}')

# Overall recommendation (V3 HMM ROC-AUC)
hmm_results = results_df[results_df['method'] == 'V3 HMM']
if len(hmm_results) > 0:
    best_hmm = hmm_results.loc[hmm_results['roc_auc'].idxmax()]
    print(f'\n>>> Recommended offset (V3 HMM ROC-AUC): {int(best_hmm["offset"])} '
          f'(AUC={best_hmm["roc_auc"]:.4f}, '
          f'{int(best_hmm["n_matched"])}/{int(best_hmm["n_known_total"])} sites matched)')

# Save
results_df.to_csv(OUT_DIR / 'offset_evaluation.csv', index=False)
print(f'\nSaved to {OUT_DIR}/')